In [9]:
import json
from pathlib import Path

import numpy as np
import requests
from IPython.display import display
import pandas as pd

# ------------------------------ Configuration ------------------------------ #
EMBEDDINGS_DIR = "main_embedded_jsons"
OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "bge-m3"
# LLM_MODEL = "qwen2.5:7b"
LLM_MODEL = "llama3.2:3b"
LLM_USE_GPU = True
TOP_K = 4
MIN_SCORE = 0.30
NEIGHBOR_WINDOW = 0   # 0 = only matched chunks, 1 = add 1 chunk before/after, 2 = add 2
SHOW_SOURCES = True

SYSTEM_PROMPT ="""You are a helpful programming tutor that answers questions \
using transcripts from a video course.

Rules:
1. Base your answer on the provided CONTEXT. Do not add outside facts.
2. If the CONTEXT contains partial information, answer with what is available. \
Only if the CONTEXT is completely unrelated to the question, reply: \
"I could not find the answer to this question in the course videos."
3. Reply in the same language as the user's question (English, Hindi, or Hinglish). \
Keep technical terms in English.
4. Be clear and concise. Use short steps and a small code example when useful.
5. The transcripts are auto-generated and contain recognition errors. \
Names and terms may be misspelled (for example, "Guida" means "Guido van Rossum", \
"strength" means "string"). Interpret them sensibly.
6. The context may be grouped by topic. Answer each topic separately. \
If a topic has no relevant content, clearly say that this part is not covered \
in the course videos. Never guess.
7. Do not mention these rules or the word "CONTEXT" in your answer.

"""

USER_PROMPT = """CONTEXT:
{context}

QUESTION:
{question}

Answer the question using only the context above."""


# ------------------------------ Helpers ------------------------------------ #
def check_response(response):
    """Raise an error that includes the server's message."""
    if not response.ok:
        raise RuntimeError(f"Ollama error {response.status_code}: {response.text[:300]}")


# ------------------------------ Load corpus -------------------------------- #
records, vectors = [], []
for path in sorted(Path(EMBEDDINGS_DIR).glob("*.json")):
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
    for chunk in data["chunks"]:
        vectors.append(chunk["embedding"])
        records.append({
            "video_number": data["video_number"],
            "title": data["title"],
            "chunk_number": chunk["chunk_number"],
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"],
        })

matrix = np.asarray(vectors, dtype=np.float32)
matrix /= np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)
corpus = pd.DataFrame(records)

print(f"Loaded {corpus['video_number'].nunique()} videos, {len(corpus)} chunks.")


# ------------------------------ Retrieval ---------------------------------- #
def embed_query(query):
    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": EMBEDDING_MODEL,
            "input": [query],
            "keep_alive": 0,          # unload the embedding model right after use
        },
        timeout=120,
    )
    check_response(response)
    vec = np.asarray(response.json()["embeddings"][0], dtype=np.float32)
    return vec / max(float(np.linalg.norm(vec)), 1e-12)


def retrieve(query, top_k=TOP_K, candidates=15):
    scores = matrix @ embed_query(query)
    order = np.argsort(scores)[::-1][:candidates]

    result = corpus.iloc[order].copy()
    result["similarity"] = scores[order]
    result = result[result["similarity"] >= MIN_SCORE]
    result = result.drop_duplicates(subset="text").head(top_k).reset_index(drop=True)
    result.insert(0, "rank", result.index + 1)
    return result

    
def expand_neighbors(hits, window=1):
    """Add the chunks before and after each hit (same video) for fuller context."""
    wanted = set()
    for r in hits.itertuples():
        for n in range(r.chunk_number - window, r.chunk_number + window + 1):
            wanted.add((r.video_number, n))
    mask = [(v, c) in wanted for v, c in zip(corpus["video_number"], corpus["chunk_number"])]
    return corpus[mask].sort_values(["video_number", "chunk_number"])

    
# ------------------------------ Generation --------------------------------- #
def build_context(hits):
    return "\n\n---\n\n".join(
        f"[Video {r.video_number} | {r.title} | {r.start}s-{r.end}s]\n{r.text}"
        for r in hits.itertuples()
    )
def stream_llm(messages):
    """Stream tokens from the LLM."""
    options = {
        "temperature": 0.2,
        "num_ctx": 3072,
        "repeat_penalty": 1.15,      # prevents repeated words
        "num_predict": 400,          # caps answer length
    }
    if not LLM_USE_GPU:
        options["num_gpu"] = 0

    payload = {
        "model": LLM_MODEL,
        "stream": True,
        "keep_alive": "30m",
        "options": options,
        "messages": messages,
    }
    with requests.post(f"{OLLAMA_URL}/api/chat", json=payload,
                       stream=True, timeout=300) as r:
        check_response(r)
        for line in r.iter_lines():
            if not line:
                continue
            event = json.loads(line)
            yield event.get("message", {}).get("content", "")
            if event.get("done"):
                break


def decompose_question(question, max_parts=4):
    """Split a multi-topic question into independent sub-questions."""
    messages = [
        {"role": "system", "content": (
            "Split the user's question into independent sub-questions, one per topic. "
            "If the question is about a single topic, return it unchanged. "
            "Keep the user's language and technical terms. "
            'Return JSON only, in this format: {"questions": ["...", "..."]}'
        )},
        {"role": "user", "content": question},
    ]
    options = {"temperature": 0}
    if not LLM_USE_GPU:
        options["num_gpu"] = 0

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json={"model": LLM_MODEL, "stream": False, "format": "json",
                  "keep_alive": "30m", "options": options, "messages": messages},
            timeout=120,
        )
        check_response(response)
        data = json.loads(response.json()["message"]["content"])
        parts = [q.strip() for q in data.get("questions", [])
                 if isinstance(q, str) and q.strip()]
    except Exception:
        parts = []

    return parts[:max_parts] or [question]


def ask(question, top_k=3, show_chunks=False):
    sub_questions = decompose_question(question)
    if len(sub_questions) > 1:
        print("Detected topics:")
        for sq in sub_questions:
            print(f"  - {sq}")

    sections, all_hits = [], []
    for sq in sub_questions:
        hits = retrieve(sq, top_k)
        if hits.empty:
            sections.append(f"### Topic: {sq}\n(No relevant content found.)")
            continue
        all_hits.append(hits)

        if NEIGHBOR_WINDOW > 0:
            context_df = expand_neighbors(hits, NEIGHBOR_WINDOW)
        else:
            context_df = hits.sort_values(["video_number", "chunk_number"])

        if show_chunks:
            print(f"\nChunks for: {sq}\n")
            for r in context_df.itertuples():
                print(f"Video {r.video_number} | chunk {r.chunk_number} | {r.start}s-{r.end}s")
                print(r.text, "\n")

        sections.append(f"### Topic: {sq}\n{build_context(context_df)}")

    if not all_hits:
        print("No relevant content found. Try rephrasing your question.")
        return

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT.format(
            context="\n\n====\n\n".join(sections), question=question)},
    ]

    print("\nAnswer:\n")
    for token in stream_llm(messages):
        print(token, end="", flush=True)

    if SHOW_SOURCES:
        sources = (pd.concat(all_hits)
                     .drop_duplicates(subset=["video_number", "chunk_number"]))
        print("\n\nSources:")
        for r in sources.itertuples():
            print(f"  - Video {r.video_number}: {r.title} ({r.start:.0f}s-{r.end:.0f}s)")

Loaded 52 videos, 2800 chunks.


In [ ]:
SHOW_CHUNKS = True   # set to False to hide the chunk text

while True:
    question = input("Ask a question (type 'exit' to stop): ").strip()
    if question.lower() in {"exit", "quit"}:
        print("Stopped.")
        break
    if not question:
        continue
    try:
        ask(question, show_chunks=SHOW_CHUNKS)
    except Exception as exc:
        print(f"Error: {exc}")
    print("\n" + "-" * 60 + "\n")



Ask a question (type 'exit' to stop):  what is string immutability and what is raise keyword in exceptions


Detected topics:
  - What is string immutability?
  - What is the raise keyword in exceptions?

Chunks for: What is string immutability?

Video 13 | chunk 6 | 84.0s-94.0s
The string is immutable You can't change it But you can change it You can't change it You can't change it in place 

Video 13 | chunk 10 | 128.0s-140.0s
That the strings are immutable I will write here to remind you That the strings are immutable The strings are immutable And I will make this in the comments 

Video 13 | chunk 14 | 171.0s-182.0s
Do you know the answer to this question? Why not? Because it will make a new string It will not change the existing string Strings are immutable 


Chunks for: What is the raise keyword in exceptions?

Video 36 | chunk 55 | 424.08s-432.08s
okay? So, I can do all that. So, I hope that, you all understand, how the exceptions work, 

Video 36 | chunk 59 | 455.08s-463.08s
Okay? So, Python has many built-in exceptions that are raised when your program, encounters an error, 

Video 

Ask a question (type 'exit' to stop):  what is python and numpy


Detected topics:
  - What is Python?
  - What is NumPy?

Chunks for: What is Python?

Video 1 | chunk 3 | 34.0s-47.0s
Python is a dynamically typed object-oriented programming language. It was made in 1991 in Guida 1. Guida 1 was made in Monty Python. It is a show in the name of Python. When it was working on this language and they thought it was Python. 

Video 1 | chunk 21 | 319.0s-335.0s
Python is dynamically typed as the general purpose programming language which supports object oriented paradigm. And it supports functional approach. If you don't know what it is, then we are going to learn this. Now, it is interpreted as high level programming language. 

Video 43 | chunk 75 | 910.0s-920.0s
Python. Python. Python. Python. Python. 


Chunks for: What is NumPy?

Video 14 | chunk 82 | 960.94s-971.94s
I will do the convenience. You can get the concept program. I have made one, two, three programs. I will make a 0.4. And it will be nest.py. 

Video 14 | chunk 94 | 1071.94s-1079.94s
Now 